# DeepSeek-MoE-16B Domain Specialization — Aggregate Routing Statistics + UMAP

Generates `deepseek_domain_specialization.json` (aggregate per-domain routing statistics:
`activation_rate`, `avg_prob`, `specialization_score`, `layer_divergence`, `domain_rate`,
`top_specialists`) plus `deepseek_domain_specialization_umap.json` (2D UMAP projection of the
same per-(layer, expert) activation-rate vectors, one dimension per domain) — the DeepSeek
counterpart of `extract_domain_specialization.ipynb`, emitting the **identical schema** so the
frontend's Domain Specialization tab reads it with no per-model special-casing.

**Same corpus as OLMoE.** The 6 domain passages below are copied verbatim from
`extract_domain_specialization.ipynb` (`code`, `math`, `biomedical`, `legal`,
`creative_writing`, `conversational`; one long ~300-400 word passage each). Only the model
differs, so any difference in the resulting statistics is a difference between the models — not
between the texts. Token *counts* still differ, since DeepSeek tokenizes the same passage
differently; that's recorded per domain in `token_counts`.

**Model config is identical to `extract_routing_trace_deepseek.ipynb`** — same exact
`transformers==4.36.2` pin, same `flash_attn` sys.modules stub, same `trust_remote_code=True`,
same `torch_dtype=` (not `dtype=`), same `EXPECTED` config check. What is *not* carried over is
the deep extraction (attention maps, expert weight downsamples, per-token expert outputs):
domain specialization only needs router logits, so `output_attentions=True` is dropped — on
~400-token passages it would materialize 27 x 16 x 400 x 400 attention maps for nothing.
`attn_implementation="eager"` is kept anyway, purely so this run uses the same attention kernel
as the trace run and the two datasets stay numerically comparable.

**DeepSeek's two divergences** (see `docs/model-architecture-jetmoe-deepseek-research.md` §3),
and how each is handled here:
1. **Layer 0 is a plain dense FFN** (no router, no experts). Its rows in every `[layer][expert]`
   array are **all zeros**, so array indices keep matching real layer numbers, and layer 0 is
   listed in the top-level `dense_layers` field. A consumer that plots these arrays blindly will
   show an empty layer 1 (1-indexed) — that is real, not missing data, and should be labelled as
   dense rather than hidden.
2. **2 shared experts fire unconditionally on every token.** They are deliberately excluded:
   firing on 100% of every domain's tokens, they carry zero discriminative signal for a
   per-domain activation-rate statistic, and including them would flatten every domain's profile
   identically. Only the 64 routed experts are counted. `shared_experts` records how many were
   left out.

Requires `trust_remote_code=True`. Run on a Colab A100 **40GB** GPU runtime (16.4B params,
~33GB in bf16). 6 forward passes total.

**Expected output size ~11-12 MB**, dominated by `expert_token_idx` (27 MoE layers x ~400 tokens
x top-6 x 6 domains ≈ 389k `[token_idx, score]` pairs, against OLMoE's 307k pairs / 8.4 MB).
That fits in git and loads as one file, but it is the largest of the three domain files — if the
Domain tab's cold load feels slow, this field is what to split or trim first (e.g. keep only the
top-N scoring tokens per expert rather than every one).

In [1]:
import importlib.util
import subprocess
import sys


def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


if importlib.util.find_spec("torch") is None:
    pip_install("torch")

# transformers PINNED TO THE EXACT VERSION -- not a range -- carried over verbatim from
# extract_routing_trace_deepseek.ipynb, where it was established the hard way. DeepSeek's
# trust_remote_code modeling_deepseek.py is 2024-era code with two confirmed hard dependencies
# on APIs that later transformers releases removed: (1) `from transformers.utils.import_utils
# import is_torch_fx_available`, present through 4.57.1, gone at the 5.0.0 major bump; (2)
# `DynamicCache.get_usable_length(...)`, called during attention's cache handling on every
# forward pass (not just multi-step generation) -- present through 4.53.0, gone by 4.55.0,
# entirely within the 4.x line. A range like ">=4.36.2,<5.0.0" lets pip resolve to the newest
# 4.x and hits break #2 immediately. Pin the EXACT version DeepSeek's own requirements.txt
# states as its floor (4.36.2) -- almost certainly what the authors tested against, and
# confirmed to have both APIs intact.
pip_install("transformers==4.36.2", "accelerate", "umap-learn", "numpy", "scikit-learn")

print("Dependency installation complete.")

Dependency installation complete.


In [2]:
import json
import os
import sys
import types
import importlib.machinery

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

# transformers 4.36.2's trust_remote_code loader (check_imports in dynamic_module_utils.py)
# does a naive static regex scan of EVERY top-level import statement in modeling_deepseek.py
# and demands each one be importable -- it only skips imports inside a try/except block, and
# flash_attn is imported under `if is_flash_attn_2_available():` (an if-guard, not try/except),
# so the scanner isn't smart enough to skip it even though it's never actually needed for a
# single eager forward pass. Stub it in sys.modules to satisfy that static scan. The stub needs
# a real __spec__ -- a bare types.ModuleType() leaves __spec__ as None, and
# importlib.util.find_spec() raises ValueError (not a clean False) when a sys.modules entry has
# __spec__ = None, which would crash transformers' own is_flash_attn_2_available() the moment it
# runs. With a real (but loader=None) ModuleSpec, find_spec() returns cleanly, and
# is_flash_attn_2_available() still correctly resolves to False afterward via
# importlib.metadata.version("flash_attn") raising PackageNotFoundError -- so the `if` guard
# still evaluates False and eager attention runs as normal; this stub cannot cause a silent
# switch to a broken flash-attn code path.
if "flash_attn" not in sys.modules:
    stub = types.ModuleType("flash_attn")
    stub.__spec__ = importlib.machinery.ModuleSpec("flash_attn", loader=None)
    sys.modules["flash_attn"] = stub

MODEL_ID = "deepseek-ai/deepseek-moe-16b-base"
OUT_PATH = "deepseek_domain_specialization.json"
UMAP_OUT_PATH = "deepseek_domain_specialization_umap.json"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Identical EXPECTED block to extract_routing_trace_deepseek.ipynb -- verify against the
# paper/config numbers before spending GPU time loading a 16B model on a wrong assumption.
EXPECTED = {
    "num_hidden_layers": 28,
    "n_routed_experts": 64,
    "num_experts_per_tok": 6,
    "n_shared_experts": 2,
    "first_k_dense_replace": 1,
    "moe_layer_freq": 1,
    "hidden_size": 2048,
    "norm_topk_prob": False,
}

config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
mismatches = {k: (getattr(config, k, "<missing>"), v) for k, v in EXPECTED.items() if getattr(config, k, None) != v}
assert not mismatches, (
    f"DeepSeek config mismatch vs. docs/model-architecture-jetmoe-deepseek-research.md: "
    f"{mismatches} -- investigate before proceeding."
)
print("DeepSeek config verified:", {k: getattr(config, k) for k in EXPECTED})

model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,  # not `dtype=` -- that alias postdates the 4.36.2 pin above; the
    # model card's own example uses torch_dtype, confirmed present in 4.36.2's from_pretrained.
    attn_implementation="eager",  # same kernel as the trace run, for numerical comparability;
    # output_attentions is NOT requested here, so no attention maps are ever materialized.
    device_map="auto",
    output_loading_info=True,
)
model.eval()

assert not loading_info["missing_keys"], (
    f"Some model weights were NOT loaded from the checkpoint (randomly initialized "
    f"instead): {loading_info['missing_keys']}"
)
print(f"unexpected_keys (informational): {loading_info.get('unexpected_keys', [])}")

num_layers = config.num_hidden_layers
hidden_size = config.hidden_size
num_experts = config.n_routed_experts        # routed experts only -- the shared ones are excluded
top_k_experts = config.num_experts_per_tok
n_shared_experts = config.n_shared_experts

# DeepseekDecoderLayer.__init__ picks DeepseekMoE (has .gate) vs. plain DeepseekMLP (dense, no
# .gate) per-layer based on first_k_dense_replace/moe_layer_freq -- read it off the
# actually-instantiated modules rather than re-deriving the condition.
moe_layer_indices = [li for li in range(num_layers) if hasattr(model.model.layers[li].mlp, "gate")]
dense_layer_indices = [li for li in range(num_layers) if li not in moe_layer_indices]
assert dense_layer_indices == [0], f"Expected only layer 0 to be dense, got {dense_layer_indices}"
print(f"Loaded {MODEL_ID}: {num_layers} layers, {len(dense_layer_indices)} dense {dense_layer_indices}, "
      f"{len(moe_layer_indices)} MoE (top-{top_k_experts} of {num_experts}, "
      f"{n_shared_experts} always-on shared experts excluded from these statistics)")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to

tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json: 0.00B [00:00, ?B/s]

...

model-00006-of-00007.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00007-of-00007.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

unexpected_keys (informational): []
Loaded deepseek-ai/deepseek-moe-16b-base: 28 layers, 1 dense [0], 27 MoE (top-6 of 64, 2 always-on shared experts excluded from these statistics)


In [3]:
# 6 domains x 1 long, coherent, grammatical passage each. All 6 passages were written from
# scratch for stylistic consistency (length, structure) -- including code/legal/biomedical,
# not just the 2 domains that are entirely new (math, conversational). The old "poetry"
# domain is retired in favor of "creative_writing". One prompt per domain keeps this to 6
# forward passes total; loading the model once is the fixed cost, prompt length barely
# affects memory, so longer passages here capture much richer per-domain routing signal.
DOMAIN_PROMPTS = {
    "code": [
        "Most production codebases begin with an interface contract before a single line of "
        "business logic gets written. A REST API endpoint is typically documented first with "
        "its request shape, response shape, and error codes, so that frontend and backend "
        "teams can work in parallel against a shared expectation rather than waiting on each "
        "other. Once the contract is settled, the backend team implements a service layer "
        "that validates input, applies business rules, and delegates persistence to a "
        "repository layer, keeping raw database queries out of the request handlers "
        "entirely.\n\n"
        "Consider a function that searches a sorted array for a target value using binary "
        "search. The function compares the target against the middle element, and if they "
        "do not match, discards the half of the array that cannot contain the target, "
        "repeating the process on the remaining half. This halves the search space on every "
        "comparison, giving binary search a logarithmic time complexity of O(log n), a "
        "dramatic improvement over the O(n) cost of scanning the array element by element, "
        "especially once the array grows into the millions of entries.\n\n"
        "Concurrency introduces its own category of bugs that rarely show up in "
        "single-threaded testing. A race condition occurs when two threads read and write "
        "shared state without proper synchronization, producing a result that depends on "
        "unpredictable timing rather than program logic. Developers guard against this with "
        "locks, atomic operations, or by redesigning the system around immutable data "
        "structures and message passing, so that no two threads ever mutate the same memory "
        "at the same time.\n\n"
        "Once a feature is implemented, it still has to survive code review and continuous "
        "integration before merging. A pull request typically triggers an automated pipeline "
        "that runs the unit test suite, checks code coverage, and lints the diff for style "
        "violations, failing the build before a human reviewer even looks at it if any of "
        "those checks do not pass. Only after the pipeline is green and a colleague has "
        "approved the change does it get merged into the main branch and queued for the "
        "next deployment."
    ],
    "math": [
        "Algebra gives us a systematic way to find unknown quantities from known "
        "relationships. To solve the equation 3x plus 7 equals 22, we isolate x by "
        "subtracting 7 from both sides to get 3x equals 15, then dividing both sides by 3 to "
        "find that x equals 5. This same principle of performing identical operations on "
        "both sides of an equation scales up to systems of many variables, which is the "
        "foundation of linear algebra and, eventually, of the matrix operations that power "
        "modern machine learning models.\n\n"
        "Calculus formalizes the idea of instantaneous change. The derivative of a function "
        "at a point measures the slope of the tangent line there, so the derivative of x "
        "cubed plus 2x with respect to x is 3x squared plus 2, telling us exactly how fast "
        "the function's output grows as x increases. Integration reverses this process, "
        "accumulating infinitely many infinitesimal slices to compute a total, such as the "
        "area under a curve or the distance traveled by an object whose velocity changes "
        "continuously over time.\n\n"
        "Geometry and number theory each contribute their own foundational facts. The "
        "Pythagorean theorem states that in a right triangle, the square of the hypotenuse "
        "equals the sum of the squares of the other two sides, a relationship that underlies "
        "everything from architectural design to GPS trilateration. A prime number, "
        "meanwhile, is a whole number greater than one that is divisible only by itself and "
        "one; the fact that every integer factors uniquely into primes is the basis of much "
        "of modern cryptography.\n\n"
        "Probability quantifies uncertainty using precise rules rather than intuition alone. "
        "If a fair six-sided die is rolled twice, the chance of rolling a six both times is "
        "one-sixth multiplied by one-sixth, or one in thirty-six, because the two rolls are "
        "independent events. This same multiplication rule, extended across thousands of "
        "variables, is what allows statisticians to model everything from election outcomes "
        "to the reliability of a manufactured part over its expected lifetime."
    ],
    "biomedical": [
        "A 58-year-old woman arrived at the emergency department reporting sudden, crushing "
        "chest pain that radiated into her jaw, along with nausea and cold sweats. An "
        "electrocardiogram showed ST-segment elevation in the anterior leads, consistent "
        "with an acute myocardial infarction, and she was taken directly to the "
        "catheterization lab, where an interventional cardiologist located and cleared a "
        "blockage in the left anterior descending artery. Within an hour of the blocked "
        "vessel being reopened, her chest pain had resolved and her cardiac enzyme levels "
        "began trending back toward normal.\n\n"
        "In a separate randomized, double-blind trial, researchers compared a new "
        "anti-inflammatory therapy against a placebo in patients with a chronic autoimmune "
        "condition. Participants who received the active treatment showed a statistically "
        "significant reduction in joint swelling and reported less pain on standardized "
        "questionnaires after twelve weeks, though a minority experienced mild "
        "injection-site irritation. The investigators concluded that the therapy was both "
        "effective and well tolerated, though they recommended a larger, multi-site "
        "follow-up trial before it could be considered for regulatory approval.\n\n"
        "At the molecular level, many of these therapies work by binding to a specific "
        "cell-surface receptor and blocking a signaling cascade that would otherwise trigger "
        "inflammation. This interrupts the release of pro-inflammatory cytokines, small "
        "proteins that normally recruit additional immune cells to a site of injury or "
        "infection, dampening the immune response without shutting it down entirely. A "
        "tissue biopsy taken before and after treatment can confirm this mechanism directly, "
        "typically showing reduced immune cell infiltration and lower levels of inflammatory "
        "markers such as C-reactive protein in the blood.\n\n"
        "Preventive medicine remains one of the most cost-effective tools available to "
        "clinicians. Routine vaccination trains the immune system to recognize a pathogen's "
        "distinctive surface proteins well before a real infection occurs, so that "
        "antibodies and memory immune cells are already circulating by the time exposure "
        "happens. Regular screening, similarly, catches conditions like hypertension or "
        "early-stage cancer while they are still asymptomatic and far easier to treat, often "
        "years before they would otherwise have produced any noticeable symptoms."
    ],
    "legal": [
        "This Master Services Agreement is entered into between the Client and the Service "
        "Provider as of the Effective Date, and governs all statements of work executed "
        "under it. The Service Provider agrees to deliver the services described in each "
        "statement of work in a professional and workmanlike manner, and the Client agrees "
        "to pay all undisputed invoices within thirty days of receipt. Either party may "
        "terminate the Agreement for convenience upon sixty days' written notice, provided "
        "that any fees accrued for work performed prior to the termination date remain due "
        "and payable in full.\n\n"
        "In a subsequent dispute, the plaintiff alleged that the defendant had breached a "
        "supply agreement by failing to deliver conforming goods by the contractually "
        "specified deadline. At trial, the court heard testimony from an industry expert "
        "regarding customary delivery timelines, together with internal correspondence in "
        "which the defendant acknowledged awareness of the deadline and its likely inability "
        "to meet it. The jury found that the defendant's failure to perform was a material "
        "breach and awarded damages calculated to place the plaintiff in the position it "
        "would have occupied had the contract been properly performed.\n\n"
        "On appeal, the defendant argued that the trial court's jury instructions on "
        "materiality were legally deficient and warranted a new trial. The appellate panel "
        "disagreed, holding that the instructions, considered in their entirety, correctly "
        "stated the governing legal standard, and that any imprecision in a single sentence "
        "did not amount to reversible error given the overwhelming weight of the evidence "
        "presented. The panel further reaffirmed that in civil actions the burden rests on "
        "the plaintiff to establish each element of the claim by a preponderance of the "
        "evidence, a standard it found comfortably satisfied on this record.\n\n"
        "Beyond contract and tort claims, corporate counsel also spend considerable time on "
        "regulatory compliance. Before launching a new product in a foreign jurisdiction, a "
        "company typically commissions a legal opinion addressing local licensing "
        "requirements, data protection obligations, and any sector-specific restrictions that "
        "might apply, since noncompliance can expose the company to fines, injunctions, or "
        "the forced withdrawal of the product from that market entirely."
    ],
    "creative_writing": [
        "The lighthouse keeper had watched a thousand storms roll in off the grey Atlantic, "
        "but something about this one made him pause at the window with his tea going cold "
        "in his hand. The waves were climbing higher than the rocks that had stood against "
        "them for three hundred years, and for the first time in his forty seasons on the "
        "island, he found himself counting the ships he could see and hoping the count would "
        "not change by morning.\n\n"
        "Mira found the letter tucked inside a hollowed-out book on her grandmother's shelf, "
        "the paper gone soft and yellow at the folds. Her hands trembled as she unfolded it, "
        "not from cold but from the particular fear of learning something that could not be "
        "unlearned, and when she finally read the first line, she understood at once why it "
        "had been hidden rather than simply thrown away.\n\n"
        "Deep in the forest, where the canopy grew so thick that noon light arrived the color "
        "of dusk, the old paths remembered every traveler who had ever walked them. The wind "
        "moved through the high branches in long, unhurried sighs, and if you stood still "
        "long enough and let your own breathing slow to match it, you could almost believe "
        "the trees were arguing quietly among themselves about whether to let you pass.\n\n"
        "By the time the last streetlamp flickered out, the city had already begun its other "
        "life, the one that belonged to the people who swept its floors and stocked its "
        "shelves while everyone else slept. A fox slipped across the empty intersection "
        "without breaking stride, entirely unbothered by the traffic lights still cycling to "
        "no one, and somewhere above the rooftops the sky was already deciding, slowly, what "
        "color the morning would be."
    ],
    "conversational": [
        "Hey, sorry for the late reply, my phone died on the way home and I didn't get a "
        "chance to charge it until just now. Anyway, are we still good for Saturday, or did "
        "something come up on your end? I can also do Sunday afternoon if that works better, "
        "just let me know so I can figure out the rest of my weekend around it.\n\n"
        "Honestly, I've been kind of exhausted this week, nothing serious, just one of those "
        "stretches where every day feels a little longer than it should. I think I just need "
        "a weekend where I don't have anywhere to be, maybe cook something simple, watch a "
        "movie I've already seen a dozen times, that kind of thing. How about you, anything "
        "fun happen lately, or has it been the same kind of week over there?\n\n"
        "Oh, that reminds me, did you end up trying that new place downtown? A couple of "
        "people at work were talking about it and apparently the line gets pretty long on "
        "weekends, so if we want to check it out we should probably go early or just do a "
        "weekday evening instead. I'm not picky either way, honestly whatever's easiest works "
        "for me, I just haven't had a good excuse to get out of the house in a while.\n\n"
        "Thanks again for helping me move that bookshelf last week, by the way, I really owe "
        "you one. Let me know if you ever need a hand with anything, moving, fixing something "
        "around the house, whatever, I'm around most weekends these days. Talk soon, and "
        "text me whenever about Saturday, no rush."
    ],
}

domains = list(DOMAIN_PROMPTS.keys())
print(f"Domains: {domains}")
for domain, prompts in DOMAIN_PROMPTS.items():
    assert len(prompts) == 1, f"{domain} has {len(prompts)} prompts, expected 1"
    print(f"  {domain}: {len(prompts[0].split())} words")


Domains: ['code', 'math', 'biomedical', 'legal', 'creative_writing', 'conversational']
  code: 341 words
  math: 326 words
  biomedical: 334 words
  legal: 349 words
  creative_writing: 293 words
  conversational: 271 words


In [4]:
# Router probabilities are recomputed from the hooked gate input rather than read off the model:
# MoEGate.forward returns only the top-k indices and their weights, not the full 64-expert
# softmax this notebook aggregates over (avg_prob needs every expert's probability, selected or
# not). Identical approach to extract_routing_trace_deepseek.ipynb. norm_topk_prob=False, so the
# scores stored alongside each token are raw softmax probabilities -- same convention as OLMoE.
def stats_for_prompt(prompt, check_against_model=False):
    """Per-layer [num_experts] top-k hit counts and summed probs, plus token count, each token's
    decoded text, and per (layer, expert) the (token index, routing score) pairs that actually
    selected it. Dense layers (no router) contribute all-zero rows and empty token lists.

    Returns (hit_counts, prob_sums, n_tokens, token_strs, expert_token_idx, reconcile) where
    reconcile is None unless check_against_model=True, in which case it is a list of
    (layer, agreement_fraction) comparing our recomputed top-k against the gate's own output."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    token_ids = inputs["input_ids"][0].tolist()
    token_strs = [tokenizer.decode([tid]) for tid in token_ids]
    n_tokens = len(token_ids)

    gate_inputs, gate_outputs = {}, {}
    hooks = []

    def make_pre_hook(li):
        def hook(module, args):
            gate_inputs[li] = args[0].detach().float().cpu()  # [1, seq, hidden]
        return hook

    def make_post_hook(li):
        def hook(module, args, output):
            gate_outputs[li] = output  # MoEGate returns (topk_idx, topk_weight, aux_loss)
        return hook

    for li in moe_layer_indices:
        hooks.append(model.model.layers[li].mlp.gate.register_forward_pre_hook(make_pre_hook(li)))
        if check_against_model:
            hooks.append(model.model.layers[li].mlp.gate.register_forward_hook(make_post_hook(li)))

    with torch.no_grad():
        model(**inputs)

    for h in hooks:
        h.remove()

    assert set(gate_inputs) == set(moe_layer_indices), (
        f"Gate hooks fired for layers {sorted(gate_inputs)}, expected {moe_layer_indices}"
    )

    # Dense layers keep all-zero rows so array index == real layer number (see the header note).
    hit_counts = torch.zeros(num_layers, num_experts)
    prob_sums = torch.zeros(num_layers, num_experts)
    expert_token_idx = [[[] for _ in range(num_experts)] for _ in range(num_layers)]
    reconcile = [] if check_against_model else None

    for li in moe_layer_indices:
        h = gate_inputs[li][0]  # [seq, hidden]
        gate_weight = model.model.layers[li].mlp.gate.weight.detach().float().cpu()  # [n_exp, hidden]
        logits = F.linear(h, gate_weight)
        assert logits.shape == (n_tokens, num_experts), (
            f"layer {li}: logits shape {tuple(logits.shape)}, expected ({n_tokens}, {num_experts})"
        )
        probs = torch.softmax(logits, dim=-1)
        topk = torch.topk(probs, k=top_k_experts, dim=-1).indices
        for t in range(n_tokens):
            hit_counts[li, topk[t]] += 1
            for e in topk[t].tolist():
                expert_token_idx[li][e].append((t, round(float(probs[t, e]), 5)))
        prob_sums[li] += probs.sum(dim=0)

        if check_against_model:
            # MoEGate's own top-k selection is the ground truth: if our recomputed softmax picks
            # the same experts, the hooked hidden state and the gate weight are both right.
            try:
                theirs = gate_outputs[li][0].detach().cpu().reshape(n_tokens, top_k_experts)
                agree = (torch.sort(theirs, dim=-1).values == torch.sort(topk, dim=-1).values).all(dim=-1)
                reconcile.append((li, float(agree.float().mean())))
            except Exception as e:  # noqa: BLE001 -- the gate's return shape isn't contractual;
                # a failure to compare is not a failure of the extraction, so report and move on.
                print(f"    [reconcile] layer {li}: could not compare against the gate output ({e!r})")

    return hit_counts, prob_sums, n_tokens, token_strs, expert_token_idx, reconcile

In [5]:
# Smoke pass on the shortest domain before the full sweep: confirms our recomputed top-k matches
# what MoEGate itself selected, that dense layer 0 stayed empty, and that routing is structurally
# sane (exactly top-k experts per token, a full softmax over all 64, not collapsed onto one
# expert -- which would mean the wrong hidden state is feeding the router).
_smoke_domain = min(DOMAIN_PROMPTS, key=lambda d: len(DOMAIN_PROMPTS[d][0]))
print(f"smoke domain: {_smoke_domain}")
_hits, _probs, _n_tok, _toks, _e_idx, _reconcile = stats_for_prompt(
    DOMAIN_PROMPTS[_smoke_domain][0], check_against_model=True)

print(f"tokens: {_n_tok}; first 8: {_toks[:8]}")
print(f"dense layer 0 hits: {_hits[0].sum().item()} (expect 0.0)")
assert _hits[0].sum().item() == 0.0, "dense layer 0 accumulated routing counts -- it has no router"
assert all(len(lst) == 0 for lst in _e_idx[0]), "dense layer 0 accumulated token indices"

_moe_hits = _hits[moe_layer_indices]
assert torch.allclose(_moe_hits.sum(dim=1), torch.full((len(moe_layer_indices),), float(top_k_experts * _n_tok))), \
    "MoE hit counts do not sum to top_k * n_tokens per layer -- top-k extraction is wrong"
assert torch.allclose(_probs[moe_layer_indices].sum(dim=1),
                      torch.full((len(moe_layer_indices),), float(_n_tok)), atol=1e-2), \
    "per-layer probabilities do not sum to n_tokens -- softmax is over the wrong axis"

_rate = _moe_hits / _n_tok
print(f"layer 1 top-8 activation rates: {sorted([round(v, 3) for v in _rate[0].tolist()], reverse=True)[:8]}")
print(f"max single-expert rate across MoE layers: {_rate.max().item():.3f}")
assert (_rate.max(dim=1).values < 0.999).any(), \
    "every layer routes ALL tokens to one expert -- the router input is almost certainly wrong"

if _reconcile:
    _agreements = [a for _, a in _reconcile]
    _worst_layer, _worst = min(_reconcile, key=lambda p: p[1])
    print(f"[reconcile] recomputed top-{top_k_experts} vs MoEGate's own selection: "
          f"mean agreement {sum(_agreements) / len(_agreements):.4f} across {len(_agreements)} layers; "
          f"worst layer {_worst_layer} at {_worst:.4f}")
    assert _worst > 0.95, (
        f"Layer {_worst_layer} only agrees with the gate's own top-k on {_worst:.2%} of tokens -- "
        f"the hooked hidden state or the gate weight path is wrong. Fix before sweeping."
    )
else:
    print("[reconcile] gate outputs were not comparable -- relying on the structural checks above.")

print("Smoke test passed.")

smoke domain: conversational
tokens: 333; first 8: ['<｜begin▁of▁sentence｜>', 'Hey', ',', ' sorry', ' for', ' the', ' late', ' reply']
dense layer 0 hits: 0.0 (expect 0.0)
layer 1 top-8 activation rates: [0.234, 0.174, 0.171, 0.165, 0.159, 0.144, 0.144, 0.135]
max single-expert rate across MoE layers: 0.703
[reconcile] recomputed top-6 vs MoEGate's own selection: mean agreement 0.9859 across 27 layers; worst layer 4 at 0.9730
Smoke test passed.


In [6]:
activation_rate = {}    # domain -> [layer][expert]  fraction of domain's tokens with expert in top-k
avg_prob = {}           # domain -> [layer][expert]  mean router softmax prob (selected or not)
token_counts = {}
prompt_counts = {}
# domain -> [token_str, ...] and domain -> [layer][expert] -> [(token_idx, score), ...] into
# that list, so the popup can show exactly which real tokens routed to a given expert/layer.
domain_tokens = {}
expert_token_idx = {}

for domain, prompts in DOMAIN_PROMPTS.items():
    print(f"\n== domain: {domain} ==")
    assert len(prompts) == 1, "domain_tokens/expert_token_idx assume exactly one prompt per domain"
    prompt = prompts[0]
    print(f"  {prompt[:80]!r}...")
    hits, probs, n_tok, token_strs, e_idx, _ = stats_for_prompt(prompt)
    activation_rate[domain] = (hits / n_tok).tolist()
    avg_prob[domain] = (probs / n_tok).tolist()
    token_counts[domain] = n_tok
    prompt_counts[domain] = len(prompts)
    domain_tokens[domain] = token_strs
    expert_token_idx[domain] = e_idx
    print(f"  total tokens: {n_tok}")


== domain: code ==
  'Most production codebases begin with an interface contract before a single line '...
  total tokens: 395

== domain: math ==
  'Algebra gives us a systematic way to find unknown quantities from known relation'...
  total tokens: 408

== domain: biomedical ==
  'A 58-year-old woman arrived at the emergency department reporting sudden, crushi'...
  total tokens: 423

== domain: legal ==
  'This Master Services Agreement is entered into between the Client and the Servic'...
  total tokens: 409

== domain: creative_writing ==
  'The lighthouse keeper had watched a thousand storms roll in off the grey Atlanti'...
  total tokens: 340

== domain: conversational ==
  "Hey, sorry for the late reply, my phone died on the way home and I didn't get a "...
  total tokens: 333


In [7]:
# Derived statistics -- formulas identical to extract_domain_specialization.ipynb. Dense layers
# carry all-zero activation rates, so every statistic below evaluates to its neutral value there
# (specialization_score log2(eps/eps) = 0, layer_divergence 0, domain_rate 0) rather than to a
# NaN or a spurious signal, and zero rates never rank into top_specialists.
EPS = 1e-4

# Synthetic baseline: none of the 6 domains is meant to be neutral/generic text, so instead of a
# 7th hand-authored "baseline" passage, use the mean activation rate across the 6 domains
# themselves, per (layer, expert), as the reference point.
baseline_rate = [
    [sum(activation_rate[d][li][e] for d in domains) / len(domains) for e in range(num_experts)]
    for li in range(num_layers)
]

# specialization_score[domain][layer][expert] = log2((rate_domain + eps) / (rate_baseline + eps))
# -- positive = over-used relative to the 6-domain average.
specialization_score = {
    d: [
        [round(float(np.log2((activation_rate[d][li][e] + EPS) / (baseline_rate[li][e] + EPS))), 4)
         for e in range(num_experts)]
        for li in range(num_layers)
    ]
    for d in domains
}

# layer_divergence[domain][layer] = total-variation distance between the domain's and the
# synthetic baseline's per-expert selection distribution (each normalized to sum to 1 across
# experts via /top_k). 0 = identical routing to the 6-domain average at that layer.
layer_divergence = {}
for d in domains:
    divs = []
    for li in range(num_layers):
        dom_dist = [activation_rate[d][li][e] / top_k_experts for e in range(num_experts)]
        base_dist = [baseline_rate[li][e] / top_k_experts for e in range(num_experts)]
        divs.append(round(0.5 * sum(abs(a - b) for a, b in zip(dom_dist, base_dist)), 5))
    layer_divergence[d] = divs

# domain_rate[domain][layer] = mean activation_rate of that domain's top-K (=top_k_experts)
# most-used experts at that layer. NOTE the cross-model caveat: K is the model's own top-k (6 for
# DeepSeek, 8 for OLMoE, 2 for JetMoE), so this is comparable across domains WITHIN a model, but
# not directly across models.
domain_rate = {
    d: [round(sum(sorted(activation_rate[d][li], reverse=True)[:top_k_experts]) / top_k_experts, 5)
        for li in range(num_layers)]
    for d in domains
}

# top experts per domain, ranked directly by real activation_rate (no baseline comparison)
top_specialists = {}
for d in domains:
    pairs = [(activation_rate[d][li][e], li, e) for li in range(num_layers) for e in range(num_experts)]
    pairs.sort(reverse=True)
    top_specialists[d] = [
        {"layer": li, "expert": e, "activation_rate": round(rate, 4)} for rate, li, e in pairs[:12]
    ]

for d in domains:
    assert domain_rate[d][0] == 0.0, "dense layer 0 should have a zero domain_rate"
    print(f"{d:>17}: domain_rate L1={domain_rate[d][1]:.3f} L{num_layers - 1}={domain_rate[d][-1]:.3f} | "
          f"top expert {top_specialists[d][0]}")

             code: domain_rate L1=0.238 L27=0.287 | top expert {'layer': 23, 'expert': 27, 'activation_rate': 0.838}
             math: domain_rate L1=0.186 L27=0.219 | top expert {'layer': 15, 'expert': 28, 'activation_rate': 0.8382}
       biomedical: domain_rate L1=0.179 L27=0.242 | top expert {'layer': 12, 'expert': 44, 'activation_rate': 0.8392}
            legal: domain_rate L1=0.181 L27=0.233 | top expert {'layer': 21, 'expert': 44, 'activation_rate': 0.7042}
 creative_writing: domain_rate L1=0.152 L27=0.193 | top expert {'layer': 19, 'expert': 40, 'activation_rate': 0.6147}
   conversational: domain_rate L1=0.175 L27=0.188 | top expert {'layer': 12, 'expert': 11, 'activation_rate': 0.7027}


In [8]:
out = {
    "domains": domains,
    "num_layers": num_layers,
    "num_experts": num_experts,
    "top_k_experts": top_k_experts,
    "token_counts": token_counts,
    "prompt_counts": prompt_counts,
    "example_prompts": DOMAIN_PROMPTS,
    "activation_rate": {d: [[round(v, 5) for v in row] for row in activation_rate[d]] for d in domains},
    "avg_prob": {d: [[round(v, 5) for v in row] for row in avg_prob[d]] for d in domains},
    "specialization_score": specialization_score,
    "layer_divergence": layer_divergence,
    "domain_rate": domain_rate,
    "domain_tokens": domain_tokens,
    "expert_token_idx": expert_token_idx,
    "top_specialists": top_specialists,
    # DeepSeek-only extras. Everything above is the plain OLMoE schema; these two say how to read
    # it: layer 0's all-zero rows are a dense FFN (not missing data), and 2 always-on shared
    # experts per MoE layer were excluded from the counts (see the header note).
    "dense_layers": dense_layer_indices,
    "shared_experts": n_shared_experts,
}

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(out, f)

print(f"\nWrote domain specialization data to {OUT_PATH} ({os.path.getsize(OUT_PATH) / 1e3:.1f} KB)")


Wrote domain specialization data to deepseek_domain_specialization.json (6176.6 KB)


## UMAP: (layer, expert) activation across domains

Reuses the `activation_rate` computed above (no extra forward passes) to build one vector per
(layer, expert) pair, one dimension per domain, and projects it to 2D with cosine-metric UMAP --
same method as `extract_domain_specialization.ipynb`. All-zero (never-activated) pairs are
excluded from the projection and reported separately as `excluded_experts`; that naturally
sweeps up all 64 of dense layer 0's placeholder rows, which is why `dense_layers` is repeated in
this file too — a consumer counting points needs to know why layer 0 contributes none.

Routed experts only: the 2 shared experts fire unconditionally on every token regardless of
domain, so they carry no discriminative signal for a per-domain activation-rate embedding.

In [9]:
import umap

NUM_LAYERS = num_layers
NUM_EXPERTS = num_experts

expert_vectors = np.array([
    [activation_rate[d][li][e] for d in domains]
    for li in range(NUM_LAYERS)
    for e in range(NUM_EXPERTS)
])

point_layer_ids = np.repeat(np.arange(NUM_LAYERS), NUM_EXPERTS)
point_expert_ids = np.tile(np.arange(NUM_EXPERTS), NUM_LAYERS)

# Never-activated (layer, expert) pairs are all-zero and undefined under the cosine metric --
# exclude from the projection, report separately as excluded_experts. This also naturally
# excludes all of dense layer 0's NUM_EXPERTS placeholder rows.
active_mask = expert_vectors.sum(axis=1) > 0
active_vectors = expert_vectors[active_mask]

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1,
                    metric="cosine", n_jobs=1)
active_embedding = reducer.fit_transform(active_vectors)

assert not np.isnan(active_embedding).any(), (
    "UMAP produced NaN coordinates even after excluding all-zero rows -- inspect "
    "active_vectors for degenerate rows, or re-run with metric='euclidean'."
)

print(f"Built {expert_vectors.shape[0]} (layer, expert) vectors across {len(domains)} domains.")
print(f"UMAP embedding shape: {active_embedding.shape} ({int(active_mask.sum())} active of {expert_vectors.shape[0]} total pairs)")

TOP_K_TOKENS = 8
active_indices = np.flatnonzero(active_mask)

umap_points = []
for row, i in enumerate(active_indices):
    layer_id = int(point_layer_ids[i])
    expert_id = int(point_expert_ids[i])
    vec = expert_vectors[i]
    dominant_domain = domains[int(np.argmax(vec))]

    # top_tokens: pool every (token, routing score) pair that selected this (layer, expert)
    # across all domains, then keep the TOP_K_TOKENS with the highest score -- so the hover popup
    # surfaces the tokens that activated this expert most strongly.
    pooled = [
        (score, domain_tokens[d][t_idx], d)
        for d in domains
        for t_idx, score in expert_token_idx[d][layer_id][expert_id]
    ]
    pooled.sort(key=lambda item: -item[0])
    top_tokens = [
        {"token": tok, "score": round(float(score), 4), "domain": d}
        for score, tok, d in pooled[:TOP_K_TOKENS]
    ]

    umap_points.append({
        "layer_id": layer_id,
        "expert_id": expert_id,
        "x": round(float(active_embedding[row, 0]), 4),
        "y": round(float(active_embedding[row, 1]), 4),
        "dominant_domain": dominant_domain,
        "domain_activation_rate": {d: round(float(vec[j]), 4) for j, d in enumerate(domains)},
        "top_tokens": top_tokens,
    })

excluded_experts = [
    {"layer_id": int(point_layer_ids[i]), "expert_id": int(point_expert_ids[i])}
    for i in np.flatnonzero(~active_mask)
]

assert len(umap_points) + len(excluded_experts) == NUM_LAYERS * NUM_EXPERTS
assert all(p["layer_id"] not in dense_layer_indices for p in umap_points), \
    "a dense layer produced a UMAP point -- it should have no routing at all"

umap_data = {
    "domains": domains,
    "num_layers": NUM_LAYERS,
    "num_experts": NUM_EXPERTS,
    "dense_layers": dense_layer_indices,
    "points": umap_points,
    "excluded_experts": excluded_experts,
}

with open(UMAP_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(umap_data, f, ensure_ascii=False, allow_nan=False, indent=2)

print(f"Wrote {UMAP_OUT_PATH} ({len(umap_points)} points, {len(excluded_experts)} excluded pairs)")

Built 1792 (layer, expert) vectors across 6 domains.
UMAP embedding shape: (1728, 2) (1728 active of 1792 total pairs)
Wrote deepseek_domain_specialization_umap.json (1728 points, 64 excluded pairs)


In [10]:
try:
    from google.colab import files
    files.download(OUT_PATH)
    files.download(UMAP_OUT_PATH)
except ImportError:
    print("Not running in Google Colab -- skipping auto-download.")
    print(f"Files were written locally at: {OUT_PATH} and {UMAP_OUT_PATH}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>